# 14 — StreamingMultiWriter: Parallel 4-Sink Fan-out

Stream generated data concurrently to all 4 destination types (Console, File, plus simulated EventHub and Kafka) using ThreadPoolExecutor fan-out.

In [ ]:
from sqllocks_spindle import Spindle, StreamingMultiWriter
from sqllocks_spindle.streaming import ConsoleSink, FileSink
import importlib, io, tempfile, os
from sqllocks_spindle.cli import _get_domain_registry

spindle = Spindle()
reg = _get_domain_registry()
mod, cls, _ = reg['retail']
domain = getattr(importlib.import_module(mod), cls)(schema_mode='3nf')

In [ ]:
# Set up 4 sinks
console_out = io.StringIO()
tmp_file = '/tmp/spindle_events.jsonl'

sink_console = ConsoleSink(file=console_out, prefix='[EVENT] ')
sink_file = FileSink(tmp_file, mode='jsonl')

# Two additional capture sinks simulating EventHub and Kafka
from sqllocks_spindle.streaming.stream_writer import StreamWriter
from typing import Any

class MockEventHubSink(StreamWriter):
    def __init__(self): self.count = 0
    def send_batch(self, events): self.count += len(events)

class MockKafkaSink(StreamWriter):
    def __init__(self): self.count = 0
    def send_batch(self, events): self.count += len(events)

mock_eh = MockEventHubSink()
mock_kafka = MockKafkaSink()
print('4 sinks ready: console, file, eventhub, kafka')

In [ ]:
# Create StreamingMultiWriter and stream
smw = StreamingMultiWriter(
    console=sink_console,
    file=sink_file,
    eventhub=mock_eh,
    kafka=mock_kafka,
    batch_size=50,
)

result = smw.stream(
    spindle.generate_stream(domain=domain, scale='small', seed=42)
)
print(result.summary())

In [ ]:
# Per-sink stats
print('\n--- Per-Sink Results ---')
for sr in result.sinks:
    print(f'  {sr.sink_name:<15} events={sr.events_sent:,}  errors={sr.error_count}')

# File sink output
file_lines = 0
try:
    with open(tmp_file) as f:
        file_lines = sum(1 for _ in f)
    print(f'\nFile sink: {file_lines} lines written to {tmp_file}')
except Exception:
    pass